# Special Topics in Software Engineering (TSIS) &mdash; Lab 1
## OpenAI API-compatible edition

**Disclaimer:** The content / instructions are the same as for the original TSIS laboratory 1 material. We've updated it to use the official [OpenAI Python API library](https://github.com/openai/openai-python) and to work with any [OpenAI API](https://openai.com/en-US/api/)-compatible LLM provider, which is pretty much all of them (in particular, [Ollama](https://ollama.com/) and [vLLM](https://vllm.ai/) for self-hosted models).

<hr/>

Welcome to the first hands-on laboratory session for the **Topici Speciale în Inginerie Software** (_Special Topics in Software Engineering_ &mdash; TSIS) course! As mentioned in the first lecture, the objective of this course is to get you up to speed on **the latest advancements** in the field of [**large language models**](https://www.ibm.com/think/topics/large-language-models) (LLMs), focusing on the impact on developing [**agentic AI**](https://agentic.ai/what-is-agentic-ai) apps.

Unlike the first generation language models (like GPT-2, FLAN-T5, BERT etc.), newer models are more than simple chatbots; they are able to **call external tools** (access files, make API calls) and **structure their output** (to output Markdown, JSON, LaTeX etc.). Combining these powerful models with the appropriate framework, a **harness**, we can turn them into self-sufficient AI agents which are able to solve practical problems.

## Running the notebook

At the lab, we will work with the [Python](https://www.python.org/) language. Since most LLM providers expose a HTTP REST API, you could in theory query them using any programming language with a HTTP client. The agentic AI framework we'll use a bit later, [LangChain](https://www.langchain.com/), has official support for Python and JavaScript/TypeScript.

Therefore, please make sure you are comfortable wih using Python for working on the labs. For the final project, it's recommended to use Python or TypeScript for the "brain" of the AI agent (although nothing prevents you from calling into tools written in other languages).

## Installing dependencies

We will use [pip](https://pip.pypa.io/en/stable/installation/) to ensure the required packages are available localy. If you're working on your own computer, you might want to make sure you have configured a [virtual environment](https://docs.python.org/3/library/venv.html) beforehand.

In [ ]:
%pip install --quiet --upgrade openai pydantic

## Setting up the OpenAI API client

We will use OpenAI's official [API library for Python](https://github.com/openai/openai-python), which can be used to query [the official OpenAI API](https://developers.openai.com/api/reference/overview). Since this API became so popular, most other inference providers have compatible endpoints.

Usually you'd sign up for an account with the inference provider and generate an API key to be able to authenticate your requests. In our case, since we use [vLLM](https://vllm.ai/) to self-host an open weight model, you can pass the value `EMPTY` for the key instead.

In [ ]:
api_key = 'EMPTY'

Let's see if it works. We'll try to get a list of the available language models.

In [ ]:
from openai import OpenAI

SELF_HOSTED_BASE_URL = "http://hpc-nodegpu01.acc-ub.local:27182/v1"
client = OpenAI(base_url=SELF_HOSTED_BASE_URL, api_key=api_key)

for model in client.models.list():
    print(model.id)

## Choosing the right model

For this lab, we will use the Qwen 3.8 model, a fast general-purpose model. See the official [model page](https://huggingface.co/Qwen/Qwen3.8-27B-FP8) on Hugging Face for more details.

We recommend you to be conservative when selecting your model, to ensure maximum efficiency and lower costs. Current generation models are quite capable, even when using a "smaller" variant. As long as you're not solving Math Olympiad problems, the Qwen 3.8 model (with 27 billion parameters) will handle itself very well!

In [ ]:
MODEL = "Qwen/Qwen3.8-27B-FP8"

## Starting a conversation

Let us start by using the [Responses API](https://developers.openai.com/api/reference/resources/responses) to "have a conversation" with the AI model. We'll keep things relatively simple: we configure a clear and concise [system prompt](https://promptengineering.org/system-prompts-in-large-language-models/), set the _thinking level_ to "low" (since we don't expect this assistant to do any sort of complicated problem-solving) and ask the LLM to answer a straightforward question.

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="You are a helpful assistant.",
    input="Explain what the thinking mode parameter does for text generation with a Large Language Model.",
    reasoning={"effort": "low"},
)

print("Response:")
print(response.output_text)

print()

usage = response.usage
if usage:
    print("--- API usage metadata ---")
    print(f"Input token count: {usage.input_tokens}")
    print(f"Output token count: {usage.output_tokens}")
    print(f"Total token count: {usage.total_tokens}")
else:
    print("No API usage information available.")

## Having a longer chat

We've seen how to send a question to the LLM and have it output a response. In practice, we want to keep a history of the messages sent between the user and the AI (the **conversation**). We call this sequence of messages a **thread**, although the Google Gen AI API uses the term **interaction**. Each user might have multiple chats, multiple threads, with an LLM, discussing different things.

Note that we set the `state` parameter of the `interactions.create` method to `False`. By default, Google's Interactions API automatically stores the state of each conversation server-side, requiring us only to provide the ID of the previous request to be able to continue the discussion. Since we want more control and we don't want our data to be stored for too long on Google's servers, we instead opt out of this behavior.

In [ ]:
class ConversationManager:
    "Manages a conversation with a language model, keeping track of the conversation history."

    def __init__(
        self,
        client: OpenAI,
        instructions: str,
        thinking_level: str = "low",
        max_history_turns: int = 3,
    ):
        self.client = client
        self.instructions = instructions
        self.thinking_level = thinking_level
        self.history = []
        self.max_messages = max_history_turns * 2
        self.model = MODEL

    def chat(self, user_input: str) -> str:
        self.history.append({"role": "user", "content": user_input})
        print(f"User input: {user_input}")

        # Trim complete user/assistant turns if exceeding the context window.
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages :]

        response = self.client.responses.create(
            model=self.model,
            # We don't store the conversation on the server side,
            # we just send the history with each request.
            store=False,
            input=self.history,
            instructions=self.instructions,
            reasoning={
                "effort": self.thinking_level,
            },
        )

        # Update the locally-stored history with the assistant's response.
        self.history += response.output

        return response.output_text

Let's test the sliding window mechanism, implemented for the conversation history.

In [ ]:
bot = ConversationManager(
    client, instructions="You are a helpful SE tutor. Be brief.", max_history_turns=2
)

print(bot.chat("My project is written in Rust and uses Actix-web."))
print()

print(bot.chat("What database library fits well with this stack?"))
print()

print(bot.chat("How do I handle connection pooling there?"))
print()

# By turn 4, turn 1 ("My project is written in Rust...") drops out of the window!
print(bot.chat("Remind me, what web framework did I say I was using?"))

Notice how the model suddenly "forgets" the information provided earlier in the conversation (as soon as we drop those messages from the conversation `history`). The full state of the conversation, including the latest user message and the previous exchanges, is known as the **context**. Current models tend to have _huge_ context windows (~1 million tokens), but sometimes even those are not large enough! We will discuss strategies for managing context later in the course.

## Structured outputs: getting structured information from raw data

One application domain for LLMs, especially in the field of Software Engineering, is **making sense out of large quantities of text data** (e.g. logs). Provided with the right **schema**, language models are able to **structure their output** to match the desired format.

Most LLM providers require the output schema to be defined using the [JSON Schema](https://json-schema.org/) format, and the OpenAI API is no different. For ease of use, however, most Python API clients support models defined using [Pydantic](https://pydantic.dev/) (which can be programatically exported as JSON schemas).

The LLM provider's backend validates the generated response and forces the model to retry if the output doesn't match the spec. This is why it's important to design flexible/permissive schemas which make sense in a given context; otherwise, the AI model might waste a lot of tokens trying to structure its output!

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class BugTriageReport(BaseModel):
    "Pydantic model for structured bug triage data extracted from raw incident reports."

    summary: str = Field(description="Concise 1-sentence technical summary of the issue")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(description="Severity level of the issue")
    affected_component: Literal["auth", "database", "frontend", "payment", "unknown"] = Field(description="Component of the system affected by the issue")
    reproduction_steps: list[str] = Field(description="Ordered list of steps to reproduce")
    requires_immediate_rollback: bool = Field(description="Whether the issue requires an immediate rollback of the deployment")

raw_ticket = """
Hey team, ever since the v2.4 deploy 10 minutes ago, users in EU cannot complete checkout!
When they click 'Pay Now' with Stripe, the spinner hangs forever and the console logs a 502 Bad Gateway
on /api/v2/charge. Tested on Chrome and Firefox. We are losing orders right now!
"""

# The OpenAI API asks the language model to output JSON matching the custom Pydantic schema.
response = client.responses.parse(
    model=MODEL,
    input="Extract structured bug triage data from raw incident reports:\n\n"
    + raw_ticket,
    text_format=BugTriageReport,
)

# Parse the JSON output into a BugTriageReport object.
triage_obj = response.output_parsed

print(f"Severity: {triage_obj.severity} | Rollback? {triage_obj.requires_immediate_rollback}")
print(f"Component: {triage_obj.affected_component}")
print(f"Steps: {triage_obj.reproduction_steps}")


## Tool calling: the application remains in control

Tool calling is not the model executing Python. The model chooses a function and supplies arguments; **our application validates and executes the function**, then sends the result back as a tool message. The model can then produce a final answer using that observation.

In [ ]:
import json


def get_service_health(service_name: str) -> str:
    "Returns the current health telemetry for an allow-listed production service as a JSON string."

    telemetry = {
        "auth-service": {
            "status": "healthy",
            "latency_ms": 42,
            "error_rate_percent": 0.01,
        },
        "payment-gateway": {
            "status": "degraded",
            "latency_ms": 1850,
            "error_rate_percent": 14.5,
        },
    }

    if service_name not in telemetry:
        raise ValueError(f"Unknown service: {service_name}")

    return json.dumps(telemetry[service_name])


service_health_declaration = {
    "type": "function",
    "name": "get_service_health",
    "description": "Read current health telemetry for an allow-listed production service.",
    "parameters": {
        "type": "object",
        "properties": {
            "service_name": {
                "type": "string",
                "enum": ["auth-service", "payment-gateway"],
            }
        },
        "required": ["service_name"],
    },
}
tools = [service_health_declaration]

history = [
    {
        "role": "user",
        "content": "Why is checkout failing? Check the payment gateway and summarize the evidence.",
    }
]

instructions = "You are an SRE assistant. Inspect telemetry before explaining incidents. Use only the provided tools."
reasoning={
    "effort": "xhigh",
}

# Call 1: the model may request a tool, but it cannot execute it itself.
response = client.responses.create(
    model=MODEL,
    store=False,
    input=history,
    instructions=instructions,
    reasoning=reasoning,
    tools=tools,
)

history += response.output

latest = response.output[-1]

while latest.type == "function_call":
    function_call = latest
    print(
        f"[application] model requested tool: {function_call.name}({function_call.arguments})"
    )

    if function_call.name != "get_service_health":
        raise ValueError(f"Unknown function call: {function_call.name}")

    print(f"[application] calling {function_call.name}({function_call.arguments})")
    arguments = json.loads(function_call.arguments)
    observation = get_service_health(**arguments)

    history.append(
        {
            "type": "function_call_output",
            "call_id": function_call.call_id,
            "output": observation,
        }
    )

    # The model receives the observation and writes the user-facing answer.
    response = client.responses.create(
        model=MODEL,
        store=False,
        input=history,
        reasoning=reasoning,
        instructions=instructions,
        tools=tools,
    )

    latest = response.output[-1]

print("\nFinal report:\n" + response.output_text)

### Exercise 2 — extend the tool safely

Add a second read-only tool, such as `get_recent_deploys()`, and expose it in the schema. Keep the implementation deterministic and allow-listed. Then ask a question that requires both tools. What happens if the model requests a tool or argument that your application does not recognize?

### Exercise 3 — mini design challenge

Design a small agent for a software engineering task (for example, a test-failure triager). Specify:

1. the system instruction and user input;
2. the structured result you would validate;
3. one or two tools, including their safety checks; and
4. one failure mode and how a human or ordinary code would detect it.

Do not implement a tool that deletes data, changes production, or sends messages.

## Takeaways

- An LLM API is a probabilistic component, not a database or an authority.
- Context is explicit: the application decides what history to send.
- Structured output gives us a machine-checkable interface, but not guaranteed truth.
- Tool calling is an application-controlled loop: request → validate → execute → observe → respond.
- Least privilege, allow-lists, timeouts, logging, and human approval matter more as tools become capable.

In the next laboratory we will build on this loop to make an agent more reliable and observable.